In [13]:
from open_dataset_store import quick_start
import pandas as pd
import numpy as np

import pandas as pd


In [14]:
store = quick_start('.', backend='local')
sum = store.summary()


Store initialised at: . (Backend: local)
📊 Dataset Store Summary
Base Directory : .
Backend        : local
------------------------------
Entities (Total: 2)
  - zones: 2
------------------------------
Entries (Total: 8)
  - experiments: 8


In [20]:
store.list_entries('experiments')

,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,processed_metadata
0,entry_0001,zone_001,1783344483,Senosr Data Test,raw_data/experiments/entry_0001_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
1,entry_0002,zone_001,1783344565,"Test CO2 variation, single person come in and ...",raw_data/experiments/entry_0002_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
2,entry_0003,zone_002,1785482758,Test 1,raw_data/experiments/entry_0005_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
3,entry_0004,zone_002,1785483118,Test 2,raw_data/experiments/entry_0006_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
4,entry_0005,zone_002,1785483802,Test 3,raw_data/experiments/entry_0005_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
5,entry_0006,zone_002,1785484346,Test 3,raw_data/experiments/entry_0006_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
6,entry_0007,zone_002,1785507025,Test 4,raw_data/experiments/entry_0007_zone_002_17855...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
7,entry_0008,zone_002,1785508506,Test 4,raw_data/experiments/entry_0008_zone_002_17855...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
8,entry_0009,zone_002,1785989318,Controller Test,raw_data/experiments/entry_0009_zone_002_17859...,{},NaN


In [16]:
csv_path = "./Collected Data/merged_data.csv"

manual_occupancy = {
    "14:20:00": 1,
    "14:30:00": 0,
}

In [17]:
import pandas as pd


df_raw = pd.read_csv(csv_path)

def add_occupancy_and_localize_time(df, occupancy_schedule, tz='Asia/Kolkata'):
    """
    Converts 'timestamp' to local time and injects a forward-filled 
    'actual_occupancy' column based on a provided manual schedule.
    """
    # 1. Ensure timestamp is timezone aware and convert to local timezone
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['timestamp'] = df['timestamp'].dt.tz_convert(tz)
    
    # 2. Create a schedule dataframe
    schedule_df = pd.DataFrame(list(occupancy_schedule.items()), columns=['time_str', 'actual_occupancy'])
    
    # Extract the base date from the dataset (assuming 1-day experiment)
    exp_date = df['timestamp'].dt.date.iloc[0]
    
    # [FIXED] Use str(exp_date) instead of exp_date.astype(str)
    schedule_df['timestamp'] = pd.to_datetime(str(exp_date) + ' ' + schedule_df['time_str'])
    schedule_df['timestamp'] = schedule_df['timestamp'].dt.tz_localize(tz)
    schedule_df = schedule_df.sort_values('timestamp')
    
    # 3. Sort main data and merge using nearest backward match (holds previous value)
    df = df.sort_values('timestamp')
    df = pd.merge_asof(df, schedule_df, on='timestamp', direction='backward')
    
    # 4. Fill any timestamps that occurred before the first schedule entry with 0
    df['actual_occupancy'] = df['actual_occupancy'].fillna(0).astype(int)
    
    return df

df_processed = add_occupancy_and_localize_time(df_raw, manual_occupancy)
display(df_processed.head())


,timestamp,outside_t,outside_h,outside_c,outside_p,room_1_t,room_1_h,room_1_c,room_1_p,room_2_t,...,ahu_oa_flow_sp,ahu_cc_temp_sp,ahu_hc_temp_sp,ahu_hum_w_sp,fan_cmnd,mixer_ratio,cooler_cmd,heater_cmd,time_str,actual_occupancy
0,2026-08-04 18:52:38.801000+05:30,28.4,79.8,425,0,24.3,88.7,0,956.9,25.9,...,0.004001,22.917135,22.917133,0.017610,40.3,61.2,1.0,0.0,14:30:00,0
1,2026-08-04 18:52:38.801000+05:30,28.4,79.8,425,0,24.3,88.7,0,956.9,25.9,...,0.004001,22.917135,22.917133,0.017610,40.3,61.2,1.0,0.0,14:30:00,0
2,2026-08-04 18:52:43.802000+05:30,28.4,79.7,411,0,24.3,88.7,0,956.9,26.0,...,0.004001,23.501028,23.501031,0.018261,40.3,61.2,1.0,0.0,14:30:00,0
3,2026-08-04 18:52:43.802000+05:30,28.4,79.7,411,0,24.3,88.7,0,956.9,26.0,...,0.004001,23.501028,23.501031,0.018261,40.3,61.2,1.0,0.0,14:30:00,0
4,2026-08-04 18:52:48.801000+05:30,28.4,79.7,422,0,24.3,88.7,0,957.0,26.1,...,0.004001,23.850102,23.850108,0.018660,40.3,61.2,1.0,0.0,14:30:00,0


In [18]:
entry_id = store.create_entry_from_df(
    entry_type="experiments",
    df=df_processed,
    entity_id="zone_002",
    description="Controller Test",
)



✅ Entry 'entry_0009' created. File saved as entry_0009_zone_002_1785989318_data.csv


In [19]:
# store.delete_entry(entry_id='entry_0007', entry_type="experiments",)